# schedule dashboard attribution refresh v1

目标：验证 `src/schedule` 的模型 dashboard refresh 能把 attribution card 传入 dashboard export，让定时刷新链路也包含模型 fidelity、window-regime 分段和模型归因摘要。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'SPEC.md').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('cannot locate repo root from notebook cwd')
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.schedule import ModelDashboardPipelineConfig, run_model_dashboard_refresh_once

CARD = Path('research/schedule_dashboard_attribution_refresh_v1')
MATRIX_CARD = CARD / 'matrix_card'
DASHBOARD_PATH = CARD / 'dashboard_export.json'
ATTRIBUTION_CARD = Path('research/model_attribution_analysis_v1')
LOCK_DIR = CARD / 'locks'
CARD.mkdir(parents=True, exist_ok=True)

{'card': str(CARD), 'attribution_card': str(ATTRIBUTION_CARD)}

{'card': 'research/schedule_dashboard_attribution_refresh_v1',
 'attribution_card': 'research/model_attribution_analysis_v1'}

## 配置刷新任务

这里不使用 mock exporter，直接使用 `run_model_dashboard_refresh_once` 的默认 matrix runner 和默认 dashboard exporter。

In [2]:
config = ModelDashboardPipelineConfig(
    name='schedule_dashboard_attribution_refresh_v1',
    card_dir=MATRIX_CARD,
    dashboard_output_path=DASHBOARD_PATH,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse'),
    top_n_factors=3,
    dashboard_top_n=5,
    attribution_card_dir=ATTRIBUTION_CARD,
    retries=1,
)
config

ModelDashboardPipelineConfig(name='schedule_dashboard_attribution_refresh_v1', card_dir=PosixPath('research/schedule_dashboard_attribution_refresh_v1/matrix_card'), dashboard_output_path=PosixPath('research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json'), start='2026-01-01', end='2026-06-09', train_end='2026-04-30', model_names=('ridge_mse', 'gcn_mse'), top_n_factors=3, dashboard_top_n=5, attribution_card_dir=PosixPath('research/model_attribution_analysis_v1'), retries=1)

In [3]:
events = []
result = run_model_dashboard_refresh_once(config, lock_dir=LOCK_DIR, notify=events.append)
run_payload = {
    'name': result.name,
    'status': result.status,
    'payload': result.payload,
    'detail': result.detail,
}
run_payload

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'name': 'schedule_dashboard_attribution_refresh_v1',
 'status': 'success',
 'payload': {'matrix_card': 'research/schedule_dashboard_attribution_refresh_v1/matrix_card',
  'dashboard_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'top_model': 'ridge_mse__top_ic_factors',
  'attribution_card': 'research/model_attribution_analysis_v1',
  'attribution_status': 'completed',
  'matrix_status': 'feature_set_matrix_completed',
  'variant_count': 6,
  'attempts': 1},
 'detail': ''}

## 校验 dashboard export

In [4]:
dashboard_payload = json.loads(DASHBOARD_PATH.read_text(encoding='utf-8'))
summary = {
    'schema_version': 'schedule.dashboard_attribution_refresh.v1',
    'status': result.status,
    'job_name': config.name,
    'attempts': result.payload.get('attempts'),
    'notifications': events,
    'matrix_card': str(MATRIX_CARD),
    'dashboard_export': str(DASHBOARD_PATH),
    'attribution_card': str(ATTRIBUTION_CARD),
    'dashboard_schema_version': dashboard_payload['schema_version'],
    'top_models': [row['model'] for row in dashboard_payload['top_models']],
    'model_fidelity': dashboard_payload['model_fidelity'],
    'window_regime_rows': len(dashboard_payload['window_regime_metrics']),
    'attribution_status': dashboard_payload['attribution']['status'],
    'attribution_models': dashboard_payload['attribution']['models'],
    'artifact_keys': sorted(dashboard_payload['artifacts'].keys()),
}
assert summary['status'] == 'success'
assert summary['attribution_status'] == 'completed'
assert summary['window_regime_rows'] > 0
assert 'attribution_top_features' in summary['artifact_keys']
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'schedule.dashboard_attribution_refresh.v1',
 'status': 'success',
 'job_name': 'schedule_dashboard_attribution_refresh_v1',
 'attempts': 1,
 'notifications': ['schedule_dashboard_attribution_refresh_v1:success'],
 'matrix_card': 'research/schedule_dashboard_attribution_refresh_v1/matrix_card',
 'dashboard_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
 'attribution_card': 'research/model_attribution_analysis_v1',
 'dashboard_schema_version': 'dashboard.model_matrix.v1',
 'top_models': ['ridge_mse__top_ic_factors',
  'gcn_mse__top_ic_factors',
  'gcn_mse__ic_weighted_composite',
  'ridge_mse__ic_weighted_composite',
  'ridge_mse__all_features'],
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 2,
  'fidelity_counts': {'native_library': 1, 'proxy_feature_adapter': 1},
  'proxy_models': ['gcn_mse'],
  'native_models': ['ridge_mse'],
  'promotion_blocker_count': 1},
 'window_regime_rows': 24,
 'attribution

In [5]:
preview = {
    'schedule_payload': result.payload,
    'dashboard_attribution': dashboard_payload['attribution'],
    'window_regime_metrics': dashboard_payload['window_regime_metrics'][:5],
    'model_effect_deltas': dashboard_payload['model_effect_deltas'][:5],
}
preview

{'schedule_payload': {'matrix_card': 'research/schedule_dashboard_attribution_refresh_v1/matrix_card',
  'dashboard_export': 'research/schedule_dashboard_attribution_refresh_v1/dashboard_export.json',
  'dashboard_schema_version': 'dashboard.model_matrix.v1',
  'top_model': 'ridge_mse__top_ic_factors',
  'attribution_card': 'research/model_attribution_analysis_v1',
  'attribution_status': 'completed',
  'matrix_status': 'feature_set_matrix_completed',
  'variant_count': 6,
  'attempts': 1},
 'dashboard_attribution': {'status': 'completed',
  'schema_version': 'analysis.model_attribution_notebook.v1',
  'models': ['ridge_mse', 'xgboost_mse', 'gcn_mse'],
  'proxy_models': ['gcn_mse'],
  'attribution_kinds': {'coefficient': None,
   'feature_importance': None,
   'permutation': None,
   'sensitivity': None},
  'top_features_by_model': [{'model': 'gcn_mse',
    'feature': 'soxx_return_5d',
    'normalized_importance': 0.6224231933249113},
   {'model': 'gcn_mse',
    'feature': 'soxl_soxx_r

## 结论

- 做了什么：通过 `src/schedule` 默认刷新流程生成模型矩阵和 dashboard export，并把 `research/model_attribution_analysis_v1` 作为 attribution card 传入。
- 核心指标：job status、attempts、通知、top model、window-regime 行数和 attribution 状态以 `analysis_summary.json` 为准。
- regime 差异：dashboard export 已包含 `window_regime_metrics` 和 `model_effect_deltas`，页面可直接按 window/regime/model 过滤。
- 是否晋升：这是框架链路验证，不是策略晋升。
- 风险：本轮仍是 paper/sim 研究链路；`gcn_mse` 仍为 proxy_feature_adapter，不能当作论文级图模型实盘结论。
- 下一步：增加 dashboard view model 或页面层筛选控件，让 window-regime、fidelity、attribution 能在交互页面中直接比较。